# Module 9 · 文字 5：案例 — IMDB 情感分析（經典 vs 現代）

> **本節定位｜整合案例**
> 把 `01`（經典 TF-IDF）與 `02–03`（現代 tokenizer + 句向量）放進**同一個真實任務**，
> 直接比較：在 IMDB 影評情感分類上，**TF-IDF + 邏輯回歸** vs **句向量 + 邏輯回歸**。
> 全程 **CPU 可跑**（用小子集 + 輕量句向量模型）。

## 學習目標
- 建立一條可重現的文本分類流程：載入 → 表示 → 分類 → 評估。
- 量化比較「稀疏 TF-IDF」與「稠密句向量」兩種特徵的表現與維度差異。
- 體會句向量的優勢（低維、語意、可遷移），並理解「想再更好 → 端到端微調」是下一步 (M11)。

<!-- concept-image:05_imdb_case_fig1 -->
<div align="center">
<img src="concept_images/05_imdb_case_fig1_pipelines_20260611_220833.png" alt="經典 vs 現代兩條文本分類管線" width="640" style="max-width:100%;">
<br><sub>圖 1 · 經典 vs 現代兩條文本分類管線</sub>
</div>

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

## 1. 載入資料：真實 IMDB 影評（HuggingFace Datasets）

使用**真實的 IMDB 電影評論資料集** `stanfordnlp/imdb`（各 25,000 篇正/負評）。
為了 CPU 友善，這裡只取一個隨機小子集做示範；換成全量即是完整訓練。

> 註：`datasets` 5.x 起，資料集需用完整的 `namespace/name`（如 `stanfordnlp/imdb`），
> 不能再用裸名 `imdb`。

In [ ]:
from datasets import load_dataset

# 載入真實 IMDB（首次執行會下載並快取）
imdb = load_dataset("stanfordnlp/imdb")
train = imdb["train"].shuffle(seed=42).select(range(400))   # 真實影評子集
test = imdb["test"].shuffle(seed=42).select(range(200))
X_train, y_train = train["text"], np.array(train["label"])
X_test, y_test = test["text"], np.array(test["label"])

print(f"資料來源：真實 IMDB（stanfordnlp/imdb）")
print(f"訓練 {len(X_train)} 篇、測試 {len(X_test)} 篇真實影評（標籤：1=正面, 0=負面）")
print(f"\n範例（截斷顯示）：\n  [{'正面' if y_train[0]==1 else '負面'}] {X_train[0][:160]}...")

## 2. 路線 A（經典）：TF-IDF + 邏輯回歸

<!-- concept-image:05_imdb_case_fig2 -->
<div align="center">
<img src="concept_images/05_imdb_case_fig2_features_20260611_182851.png" alt="稀疏 TF-IDF vs 稠密句向量的特徵空間對比" width="640" style="max-width:100%;">
<br><sub>圖 2 · 稀疏 TF-IDF vs 稠密句向量的特徵空間對比</sub>
</div>

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)
print(f"TF-IDF 特徵維度 = {Xtr_tfidf.shape[1]}（稀疏）")

clf_a = LogisticRegression(max_iter=1000)
clf_a.fit(Xtr_tfidf, y_train)
pred_a = clf_a.predict(Xte_tfidf)
acc_a, f1_a = accuracy_score(y_test, pred_a), f1_score(y_test, pred_a)
print(f"[TF-IDF]   Accuracy = {acc_a:.3f} | F1 = {f1_a:.3f}")

## 3. 路線 B（現代）：句向量 + 邏輯回歸

用 `sentence-transformers` 的輕量模型把每篇影評編成 384 維稠密句向量，再用同樣的邏輯回歸分類。

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    Xtr_emb = st.encode(X_train, show_progress_bar=False)   # (N_train, 384)
    Xte_emb = st.encode(X_test, show_progress_bar=False)
    print(f"句向量特徵維度 = {Xtr_emb.shape[1]}（稠密，僅 384 維！）")

    clf_b = LogisticRegression(max_iter=1000)
    clf_b.fit(Xtr_emb, y_train)
    pred_b = clf_b.predict(Xte_emb)
    acc_b, f1_b = accuracy_score(y_test, pred_b), f1_score(y_test, pred_b)
    print(f"[句向量]   Accuracy = {acc_b:.3f} | F1 = {f1_b:.3f}")
except Exception as e:
    acc_b = f1_b = None
    print("（未安裝 sentence-transformers，略過路線 B）", e)

## 4. 比較與解讀

<!-- concept-image:05_imdb_case_fig3 -->
<div align="center">
<img src="concept_images/05_imdb_case_fig3_performance_20260611_220914.png" alt="性能對比：準確度與維度權衡" width="640" style="max-width:100%;">
<br><sub>圖 3 · 性能對比：準確度與維度權衡</sub>
</div>

In [ ]:
print(f"{'方法':<14}{'特徵維度':<12}{'Accuracy':<10}{'F1':<8}")
print(f"{'TF-IDF':<14}{Xtr_tfidf.shape[1]:<12}{acc_a:<10.3f}{f1_a:<8.3f}")
if acc_b is not None:
    print(f"{'句向量(384)':<14}{Xtr_emb.shape[1]:<12}{acc_b:<10.3f}{f1_b:<8.3f}")

print("""
觀察重點：
- TF-IDF 用了數千維稀疏特徵；句向量只用 384 維稠密特徵就能達到相當（常更好）的表現。
- 句向量帶語意：對同義改寫、罕見措辭更穩健，且可直接遷移到檢索/分群等其他任務。
- 在小資料上兩者可能接近；資料越大、語言越自然，現代嵌入的優勢越明顯。
""")

## 小結與延伸

| 路線 | 特徵 | 維度 | 特性 |
|:--|:--|:--|:--|
| 經典 | TF-IDF | 數千（稀疏） | 強基線、可解釋，但上下文無關 |
| 現代 | 句向量 | 384（稠密） | 語意、低維、可遷移 |

**想要更高的天花板？** 以上是「**凍結特徵 + 傳統分類器**」。
若把 Transformer **端到端微調**（連同它的權重一起學），通常能再往上一截——
這正是 **Module 11 · `02_text_downstream`** 要做的：用這份 tokenized 資料微調一個
DistilBERT 分類器，並介紹 LLM 的 LoRA/SFT 微調。